In [ ]:
import sys
print('Python: {}'.format(sys.version))
import scipy
print('scipy: {}'.format(scipy.__version__))
import numpy
print('numpy: {}'.format(numpy.__version__))
import matplotlib
print('matplotlib: {}'.format(matplotlib.__version__))
import pandas as pd
import sklearn
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import numpy as np
from sklearn import datasets
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
import seaborn as sns
from sklearn.metrics import classification_report
plt.rcParams["figure.dpi"] = 800

In [ ]:
def loadData(fileName):
  return pd.read_excel(fileName)

In [ ]:
def loadClassifiers():
  C = 10
  kernel = 1.0 * RBF([1.0, 1.0])  # for GPC
  return {
    "Gaussian NB": GaussianNB(),
    "Logistic Regression": LogisticRegression(multi_class="multinomial"),
    'L1 Logistic': LogisticRegression( penalty='l1', solver='saga', multi_class='multinomial'),
    'L2 Logistic ': LogisticRegression(penalty='l2', solver='saga', multi_class='multinomial'),
    "Random Forest": RandomForestClassifier(),
    }

In [ ]:
def dataClassify(data):
  return data.drop(columns=['RevenueLossbyUpgrade', 'RevenuebyStays', 'RevenueLossbyCancel','PotentialRevenue','CustomerType','AssignedRoom','ReservedRoom'])

In [ ]:
def isLastcancel(arrLike):
  return str(1) if(arrLike['ReservationStatus']=='Canceled' and ((arrLike['ArrivalDate']-arrLike['ReservationStatusDate']).days<=3)) else str(0)

In [ ]:
def getTrainSet(data):
  dataSet=dataClassify(data)
  dataSet['isLastCancel']=dataSet.apply(isLastcancel,axis=1)
  Y_train=dataSet[['isLastCancel']]
  X_train=dataSet.drop(columns=['isLastCancel','ReservationStatus','ArrivalDate','ReservationStatusDate',])
  catlabels=['BookingDistributionChannel','Country','DepositType','MarketSegment','BookingMeal',]
  for i in catlabels:
    X_train[i] = X_train[i].astype('category')
  cat_columns = X_train.select_dtypes(['category']).columns
  X_train[cat_columns] = X_train[cat_columns].apply(lambda x: x.cat.codes)
  return [X_train.to_numpy(),list(Y_train['isLastCancel'])]

In [ ]:
def classificationReport(true,pred,target_names,title,ax):
  clf_report = classification_report(true,pred,labels=np.arange(2),target_names=target_names,output_dict=True)
  ax.set_title(title)
  sns.heatmap(pd.DataFrame(clf_report).iloc[:-1, :].T, annot=True,ax=ax)

In [ ]:
def fitClassify(x,y,classifiers):
  fitted=[]
  N=len(classifiers)
  fig, axes = plt.subplots(1, N, sharex=True,figsize=(30,15))
  fig.suptitle('Classifiers Comparison')
  count=0
  for classifier_idx, (name, classifier) in enumerate(classifiers.items()):
    tmp=classifier.fit(x, y)
    fitted.append(tmp)
    y_pred = tmp.predict(x)
    accuracy = accuracy_score(y, y_pred)
    print(f"Accuracy (train) for {name}: {accuracy:0.1%}")
    classificationReport(y,y_pred,['Last-minute Cancellations'if i=='1' else 'Non Cancellations' for i in tmp.classes_],name,axes[count])
    count+=1
  plt.savefig("Cancel Model.png",transparent=True)
  plt.show()
  return fitted

In [ ]:
def getProba(bestFitted,x):
  to_return=pd.DataFrame(bestFitted.predict_proba(x),columns =['Last-minute Cancellations'if i==1 else 'Non Cancellations' for i in bestFitted.classes_])
  to_return.to_excel("Cancellationse_proba.xlsx",index=False)
  return to_return

In [ ]:
def calCancelPpremium(probaMatrix,dataset):
  dataset['CancelPremium']=np.multiply(probaMatrix[:,-1],((data['StaysInWeekendNights']+data['StaysInWeekNights']-1)*data['AverageDailyRate']).to_numpy().T)
  dataset['CancelPremium']=dataset['CancelPremium'].clip(lower=0)
  dataset['CancelPremium']=dataset['CancelPremium']/(dataset['StaysInWeekendNights']+dataset['StaysInWeekNights'])
  dataset['CancelPremium']=dataset['CancelPremium'].replace(np.nan, 0)
  return dataset

In [ ]:
data=loadData('R2_final_hotels_Clean.xlsx')

In [ ]:
classTrainSet=getTrainSet(data)

In [ ]:
fittedModel=fitClassify(classTrainSet[0],classTrainSet[1],loadClassifiers())

In [ ]:
getProba(fittedModel[-1],classTrainSet[0])

In [ ]:
probaMatrix=pd.read_excel("Cancellationse_proba.xlsx").to_numpy()

In [ ]:
data_cancelpremium=calCancelPpremium(probaMatrix,data)

In [ ]:
data_cancelpremium.to_excel("R2_final_hotels_cancelpremium.xlsx",index=False)